# NBA Win Probability — Exploratory Data Analysis

Skeleton EDA notebook for Phase 1 data validation. Run after all three fetch modules complete.

## 1. Setup & DB connections

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

conn_games   = sqlite3.connect('../data/raw/games.db')
conn_pbp     = sqlite3.connect('../data/raw/pbp.db')
conn_players = sqlite3.connect('../data/raw/players.db')

print('Connections open.')

## 2. Game log overview

In [ ]:
# Game counts by season (should be ~1,230 per regular season)
counts = pd.read_sql(
    "SELECT season, COUNT(DISTINCT game_id) as games FROM game_logs GROUP BY season ORDER BY season",
    conn_games
)
print(counts.to_string(index=False))

In [ ]:
# Home win rate by season
home_wl = pd.read_sql(
    "SELECT season, wl, COUNT(*) as n FROM game_logs WHERE is_home=1 GROUP BY season, wl ORDER BY season",
    conn_games
)
print(home_wl)

In [ ]:
# Score distributions
scores = pd.read_sql("SELECT season, pts FROM game_logs WHERE pts IS NOT NULL", conn_games)
fig, ax = plt.subplots(figsize=(10, 4))
for season, grp in scores.groupby('season'):
    grp['pts'].plot.kde(ax=ax, label=season)
ax.set_xlabel('Points scored')
ax.set_title('Score distribution by season')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Team efficiency distributions

In [ ]:
eff = pd.read_sql("SELECT * FROM team_efficiency", conn_games)
print(f"Rows: {len(eff)}, Teams per season: {eff.groupby('season')['team_id'].nunique().to_dict()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes, ['off_rating', 'def_rating', 'pace'], ['OffRtg', 'DefRtg', 'Pace']):
    sns.boxplot(data=eff, x='season', y=col, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 teams by net rating (most recent season)
latest = eff[eff['season'] == eff['season'].max()].sort_values('net_rating', ascending=False)
print('Top 5 by net rating:', latest[['team_name', 'off_rating', 'def_rating', 'net_rating']].head().to_string(index=False))
print('Bottom 5:', latest[['team_name', 'off_rating', 'def_rating', 'net_rating']].tail().to_string(index=False))

## 4. Play-by-play event type breakdown

In [ ]:
event_counts = pd.read_sql(
    "SELECT action_type, COUNT(*) as n FROM play_by_play GROUP BY action_type ORDER BY n DESC LIMIT 15",
    conn_pbp
)
print(event_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=event_counts, x='n', y='action_type', ax=ax)
ax.set_title('Top 15 play-by-play event types')
ax.set_xlabel('Event count')
plt.tight_layout()
plt.show()

## 5. Score differential over time (single game example)

In [ ]:
# Replace with any game_id from your database
# To find an interesting game: pd.read_sql("SELECT DISTINCT game_id FROM play_by_play LIMIT 5", conn_pbp)
EXAMPLE_GAME_ID = pd.read_sql("SELECT DISTINCT game_id FROM play_by_play LIMIT 1", conn_pbp)['game_id'].iloc[0]
print(f"Using game_id: {EXAMPLE_GAME_ID}")

pbp = pd.read_sql(
    "SELECT period, action_number, clock_seconds, score_home, score_away, action_type, description "
    "FROM play_by_play WHERE game_id=? ORDER BY period, action_number",
    conn_pbp, params=(EXAMPLE_GAME_ID,)
)

# Carry-forward scores (NULL on non-scoring plays)
pbp['score_home'] = pbp['score_home'].ffill().fillna(0).astype(int)
pbp['score_away'] = pbp['score_away'].ffill().fillna(0).astype(int)
pbp['diff'] = pbp['score_home'] - pbp['score_away']
pbp['event_seq'] = range(len(pbp))

print(f"Total events: {len(pbp)}")
pbp.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pbp['event_seq'], pbp['diff'], linewidth=1)
ax.axhline(0, color='black', linewidth=0.5)

# Period boundaries
for period in pbp['period'].unique():
    first_event = pbp[pbp['period'] == period]['event_seq'].iloc[0]
    ax.axvline(first_event, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.text(first_event, ax.get_ylim()[1] * 0.9, f'Q{period}', fontsize=8, color='gray')

ax.set_xlabel('Event sequence')
ax.set_ylabel('Score differential (home − away)')
ax.set_title(f'Score differential over time — game {EXAMPLE_GAME_ID}')
plt.tight_layout()
plt.show()

## 6. Player box score distributions

In [ ]:
players = pd.read_sql(
    "SELECT game_id, player_id, player_name, minutes, pts, ast, reb, tov, fg_pct FROM player_box_scores LIMIT 50000",
    conn_players
)
print(f"Sample rows: {len(players)}")

# Parse minutes to float for filtering
def parse_minutes(m):
    if not m or ':' not in str(m):
        return 0.0
    parts = str(m).split(':')
    return int(parts[0]) + int(parts[1]) / 60

players['min_float'] = players['minutes'].apply(parse_minutes)
dnp = players[players['min_float'] == 0]
active = players[players['min_float'] > 0]
print(f"DNP (0 min) rows: {len(dnp)} — will exclude in Phase 2 feature engineering")
print(f"Active player rows: {len(active)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['pts', 'ast', 'reb']):
    active[col].dropna().plot.hist(ax=ax, bins=30, edgecolor='white')
    ax.set_title(col.upper())
    ax.set_xlabel('Per game')
plt.suptitle('Player box score distributions (active players only)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Close connections
conn_games.close()
conn_pbp.close()
conn_players.close()
print('Done.')